In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import csv
import math
import statistics
import pickle
from itertools import combinations
import os
import heapq
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
g = nx.read_graphml("weightedGraph_adult.graphml")

In [ ]:
for x in g.nodes():
    print(x)
    break

In [ ]:
with open('Adulta/colonists_2.5.pickle', 'rb') as f:
    colonists = pickle.load(f)

In [ ]:
coloni = []
colonie = []

for k, v in colonists.items():
    coloni.append(k)
    colonie.append(v)

In [ ]:
egos = []
for colono in coloni:
    ego = nx.ego_graph(g, str(colono))
    egos.append(ego)

### Creazione file csv per colonie e coloni

In [ ]:
for i, e in enumerate(egos):
    with open('ColonieLarvaGephi/colonie/colony' + str(i) + '.csv', mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        # Scrivi l'intestazione delle colonne
        writer.writerow(['id', 'celltype', 'additional_annotations', 'level_7_cluster', 'x', 'y', 'z', 'hemisphere', 'colony'])
        for node, attributes in e.nodes(data=True):
            colony = -1
            j = 0
            found = False
            for c in colonie[i]:
                if int(node) in c:
                    colony = j
                    found = True
                j += 1
            if node == str(coloni[i]):
                colony = -100
    
            writer.writerow([node, attributes['celltype'], attributes['additional_annotations'], attributes['level_7_cluster'], 
                             attributes['x'], attributes['y'], attributes['z'], attributes['hemisphere'], colony])

In [ ]:
for i, e in enumerate(egos):
    with open('ColonieLarvaGephi/colonie/colony' + str(i) + '_edges.csv', mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        edges = e.edges()
        writer.writerow(['source', 'target'])
        for edge in edges:
            writer.writerow([edge[0], edge[1]])

### Scrittura file info tot

In [ ]:
with open('ColonieAdultaGephi/total_info.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['type', '#nodes', '#edges', 'density',
                     '#celltype', 'transitivity', 'attribute_assortativity_coefficient',
                     'weighted_degree_assortativity', 'degree_assortativity', 'avg_weighted_neighbor_degree',
                     'avg_neighbor_degree'])
    for i, e in enumerate(egos):
        writer.writerow(['ego', -1, -1, -1,
                         -1, -1, -1, -1, -1,
                         -1, -1])

        j = 1
        for c in colonie[i]:
            strings = [str(x) for x in c]
            sub = g.subgraph(strings)
            nodes = sub.number_of_nodes()
            edges = sub.number_of_edges()
            density = nx.density(sub)
            s = set()
            for _, data in sub.nodes(data=True):
                s.add(data['superclass'])
            celltype = len(s)
            transitivity = nx.transitivity(sub)
            attr_assort_coeff = nx.attribute_assortativity_coefficient(sub, 'superclass')
            deg_weight_assort = nx.degree_assortativity_coefficient(sub, weight='weight')
            deg_assort = nx.degree_assortativity_coefficient(sub)
            avg_weight_neigh_deg = nx.average_neighbor_degree(sub, weight='weight')
            avg_neigh_deg = nx.average_neighbor_degree(sub)
            writer.writerow(['colony' + str(j), nodes, edges, density,
                             celltype, transitivity, attr_assort_coeff, deg_weight_assort, deg_assort,
                             avg_weight_neigh_deg, avg_neigh_deg])
            j += 1

In [ ]:
def global_efficiency(graph):
    nodes = graph.nodes()
    N = len(nodes)
    
    if N < 2:
        return 0
    
    efficiency = 0
    for source in nodes:
        for target in nodes:
            if source != target:
                try:
                    dist = nx.shortest_path_length(graph, source=source, target=target)
                    efficiency += 1 / dist
                except nx.NetworkXNoPath:
                    continue
    
    return efficiency / (N * (N - 1))

def graph_entropy(graph):
    total_degree = sum(dict(graph.degree()).values())
    
    if total_degree == 0:
        return 0
    
    probs = [deg / total_degree for deg in dict(graph.degree()).values()]

    entropy = -sum(p * np.log(p) for p in probs if p > 0)
    return entropy

def clustering_coefficient_graph(graph):
    local_clustering = nx.clustering(graph)
    global_clustering = sum(local_clustering.values()) / len(graph.nodes()) if graph.nodes() else 0
    
    return global_clustering

In [ ]:
with open('ColonieAdultaGephi/total_info_2.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['type', 'weakly_connected_components', 'biggest_wcc', 'strongly_connected_components', 'biggest_scc',
                     'avg_distances', 'global_efficiency', 'entropy', 'clustering_coefficient'])
    for i, e in enumerate(egos):
        writer.writerow(['ego', -1, -1, -1, -1, -1, -1, -1, -1])

        j = 1
        for c in colonie[i]:
            strings = [str(x) for x in c]
            sub = g.subgraph(strings)
            wcc_list = list(nx.weakly_connected_components(sub))
            wcc = len(wcc_list)
            biggest_wcc = max(len(comp) for comp in wcc_list)
            scc_list = list(nx.strongly_connected_components(sub))
            scc = len(scc_list)
            biggest_scc = max(len(comp) for comp in scc_list)
            s = 0
            cnt = 0
            for _, _, data in sub.edges(data=True):
                s += data['weight']
                cnt += 1
            avg_distances = s/cnt if cnt != 0 else 0
            glob_eff = global_efficiency(sub)
            entropy = graph_entropy(sub)
            clustering_coefficient = clustering_coefficient_graph(sub)
            
            writer.writerow(['colony' + str(j), wcc, biggest_wcc, scc, biggest_scc, avg_distances, glob_eff, entropy, clustering_coefficient])
            j += 1

### Preprocessing

In [ ]:
data = pd.read_csv('ColonieAdultaGephi/total_info.csv')

In [ ]:
import json

def media(d):
    try:
        s = json.loads(d.replace("'", '"'))
        return sum(list(s.values())) / len(s)
    except AttributeError:
        return d

In [ ]:
type_list = list(data["type"])
contatore = -1
index_list = []
for t in type_list:
    if 'ego' in t:
        contatore += 1
    index_list.append(contatore)

data['node'] = index_list

data['node'] = data['node'].apply(lambda x: str(coloni[x]))

In [ ]:
data['avg_weighted_neighbor_degree'] = data['avg_weighted_neighbor_degree'].apply(lambda x: media(x))

In [ ]:
data['avg_weighted_neighbor_degree']

In [ ]:
data['avg_neighbor_degree'] = data['avg_neighbor_degree'].apply(lambda x: media(x))

In [ ]:
data['avg_neighbor_degree']

In [ ]:
data.fillna(0, inplace=True)
data = data.loc[(data["type"] == "colony1") | (data["type"] == "colony2")][[
    'node', '#edges', 'density', 'transitivity',
    'attribute_assortativity_coefficient', 'weighted_degree_assortativity',
    'degree_assortativity', 'avg_weighted_neighbor_degree', 'avg_neighbor_degree'
]].reset_index(drop=True)

data.set_index('node', inplace=True)

In [ ]:
data.to_csv('ColonieAdultaGephi/for_clustering.csv')

data

### Preprocessing 2


In [ ]:
data = pd.read_csv('ColonieAdultaGephi/total_info_2.csv')

In [ ]:
type_list = list(data["type"])
contatore = -1
index_list = []
for t in type_list:
    if 'ego' in t:
        contatore += 1
    index_list.append(contatore)

data['node'] = index_list

data['node'] = data['node'].apply(lambda x: str(coloni[x]))

In [ ]:
data.fillna(0, inplace=True)
data = data.loc[(data["type"] == "colony1") | (data["type"] == "colony2")][[
    'node', 'weakly_connected_components', 'biggest_wcc', 'strongly_connected_components', 'biggest_scc',
    'avg_distances', 'global_efficiency', 'entropy', 'clustering_coefficient'
]].reset_index(drop=True)

data.set_index('node', inplace=True)

In [ ]:
data.to_csv('ColonieAdultaGephi/for_clustering_2.csv')

data

### Clustering Enri

In [ ]:
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import MinMaxScaler

features = [
    '#edges', 'density', 'transitivity',
    'attribute_assortativity_coefficient', 'weighted_degree_assortativity',
    'degree_assortativity', 'avg_weighted_neighbor_degree', 'avg_neighbor_degree'
]

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data[features])

dbscan = DBSCAN(eps=0.27, min_samples=5)
data['cluster'] = dbscan.fit_predict(scaled_data)

# View clusters
print(data['cluster'].value_counts())

In [ ]:
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import numpy as np

# Compute the distances to the k-th nearest neighbor
k = 5  # k = min_samples
neigh = NearestNeighbors(n_neighbors=k)
distances, indices = neigh.fit(scaled_data).kneighbors(scaled_data)

# Sort and plot distances to find the "elbow"
distances = np.sort(distances[:, -1])
plt.plot(distances)
plt.title("k-Distance Plot")
plt.xlabel("Data Points (sorted)")
plt.ylabel(f"Distance to {k}-th Nearest Neighbor")
plt.show()

In [ ]:
data

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Extract the features and exclude 'node' for clustering
features = data.drop(columns=['node'])
node_ids = data['node']

# Standardize the data
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Determine the optimal number of clusters using the Elbow Method
inertia = []
k_values = range(2, 10)  # Testing for k = 2 to 9

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(scaled_features)
    inertia.append(kmeans.inertia_)

# Plot the Elbow Curvedata
plt.figure(figsize=(8, 6))
plt.plot(k_values, inertia, marker='o', linestyle='-')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.xticks(k_values)
plt.grid()
plt.show()

# Calculate silhouette scores for each k to further analyze cluster quality
silhouette_scores = []
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42)
    cluster_labels = kmeans.fit_predict(scaled_features)
    silhouette_scores.append(silhouette_score(scaled_features, cluster_labels))

# Plot silhouette scores
plt.figure(figsize=(8, 6))
plt.plot(k_values, silhouette_scores, marker='o', linestyle='-')
plt.title('Silhouette Scores for k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.xticks(k_values)
plt.grid()
plt.show()

In [ ]:
# Perform K-Means clustering with k=4
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
cluster_labels = kmeans.fit_predict(scaled_features)

# Add cluster labels to the original dataframe
data['cluster'] = cluster_labels
data


In [ ]:
# Create scatter plots for each pair of features, colored by cluster, and save them in a single figure
features = [
    '#edges', 'density', 'transitivity',
    'attribute_assortativity_coefficient', 'weighted_degree_assortativity',
    'degree_assortativity', 'avg_weighted_neighbor_degree', 'avg_neighbor_degree'
]

# Create a grid of scatter plots
num_features = len(features)
fig, axes = plt.subplots(
    nrows=num_features, ncols=num_features,
    figsize=(35, 35), sharex='col', sharey='row'  # Larger figsize for better visibility
)

# Loop through each pair of features
for i, feature1 in enumerate(features):
    for j, feature2 in enumerate(features):
        ax = axes[i, j]
        ax.set_xscale('log')
        ax.set_yscale('log')
        if i == j:  # Diagonal: Plot histograms
            ax.hist(df[feature1], bins=20, color='gray', alpha=0.7)
            ax.set_title(feature1, fontsize=12, rotation=90)
        else:  # Off-diagonal: Scatter plots
            scatter = ax.scatter(
                df[feature2], df[feature1],
                c=df['cluster'], cmap='viridis', s=15, alpha=0.7
            )
        if i == num_features - 1:
            ax.set_xlabel(feature2, fontsize=10, rotation=90)
        if j == 0:
            ax.set_ylabel(feature1, fontsize=10)

# Add a legend and colorbar to explain cluster colors
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(scatter, cax=cbar_ax)
cbar.set_label('Cluster', fontsize=14)

# Adjust layout and save to a file
plt.tight_layout(rect=[0, 0, 0.9, 1])
output_figure_path = 'feature_scatterplots_kmeans_loglog.png'
plt.savefig(output_figure_path, dpi=300)
plt.show()

output_figure_path


In [ ]:
import os

import numpy as np

features = [
    '#edges', 'density', 'transitivity',
    'attribute_assortativity_coefficient', 'weighted_degree_assortativity',
    'degree_assortativity', 'avg_weighted_neighbor_degree', 'avg_neighbor_degree'
]

# Remove outliers
df_no_outliers = remove_outliers_iqr(df.copy(), features)

# Save scatter plots again without outliers
output_dir_no_outliers = 'scatterplots_clusters_log'
os.makedirs(output_dir_no_outliers, exist_ok=True)

for i, feature1 in enumerate(features):
    for j, feature2 in enumerate(features):
        if i != j:  # Skip diagonal
            plt.figure(figsize=(8, 6))
            plt.scatter(
                df_no_outliers[feature2].replace(0, 1e-9),  # Avoid log(0)
                df_no_outliers[feature1].replace(0, 1e-9),
                c=df_no_outliers['cluster'], cmap='viridis', s=10, alpha=0.7
            )
            plt.xscale('log')
            plt.yscale('log')
            plt.title(f'Log-Log Scatter Plot (No Outliers): {feature1} vs {feature2}', fontsize=14)
            plt.xlabel(feature2, fontsize=12)
            plt.ylabel(feature1, fontsize=12)
            plt.colorbar(label='Cluster')
            plt.grid(alpha=0.3, which='both', linestyle='--')

            # Save the plot
            plot_path = os.path.join(output_dir_no_outliers, f'log_{feature1}_vs_{feature2}_no_outliers.png')
            plt.savefig(plot_path, dpi=300)
            plt.close()

output_dir_no_outliers


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

# Directory containing the saved log-log scatter plots
output_dir_log = 'scatterplots_clusters_log'

# Get the list of all saved plots
plot_files = [f for f in os.listdir(output_dir_log) if f.endswith('.png')]

# Determine the grid size for subplots
num_plots = len(plot_files)
cols = 3  # Number of columns
rows = (num_plots + cols - 1) // cols  # Compute rows needed

# Create a figure with a grid of subplots
fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5))

# Flatten axes for easy indexing
axes = axes.flatten()

# Loop through each plot file and add to the grid
for idx, plot_file in enumerate(plot_files):
    img_path = os.path.join(output_dir_log, plot_file)
    img = Image.open(img_path)
    axes[idx].imshow(img)
    axes[idx].axis('off')  # Hide axes
    axes[idx].set_title(plot_file, fontsize=10)

# Hide any remaining empty subplots
for ax in axes[len(plot_files):]:
    ax.axis('off')

# Add a main title
fig.suptitle('Log-Log Scatter Plots for Feature Pairs', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.95)

# Save the combined figure
combined_output_path = 'combined_scatterplots_log.png'
plt.savefig(combined_output_path, dpi=300)
combined_output_path


### Clustering Chris - Expectation Maximization

In [ ]:
data = pd.read_csv('ColonieLarvaGephi/for_clustering.csv')

In [ ]:
data = data.drop(columns=['Unnamed: 0'])

In [ ]:
data

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

features = [
    '#edges', 'weighted_degree_assortativity', 'avg_weighted_neighbor_degree', 'avg_neighbor_degree'
]

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data[features])

In [ ]:
from sklearn.mixture import GaussianMixture
import numpy as np
import matplotlib.pyplot as plt

bic_scores = []
aic_scores = []
n_components_range = range(1, 11)
for n in n_components_range:
    gmm = GaussianMixture(n_components=n, random_state=42)
    gmm.fit(scaled_data)
    bic_scores.append(gmm.bic(scaled_data))
    aic_scores.append(gmm.aic(scaled_data))

plt.figure(figsize=(8, 4))
plt.plot(n_components_range, bic_scores, label='BIC', marker='o')
plt.plot(n_components_range, aic_scores, label='AIC', marker='o')
plt.xlabel('Numero di cluster')
plt.ylabel('Score')
plt.legend()
plt.title('Selezione del numero di cluster')
plt.show()

In [ ]:
optimal_bic_n = n_components_range[np.argmin(bic_scores)]
optimal_aic_n = n_components_range[np.argmin(aic_scores)]

print(f"Numero ottimale di cluster (BIC): {optimal_bic_n}")
print(f"Numero ottimale di cluster (AIC): {optimal_aic_n}")

In [ ]:
gmm = GaussianMixture(n_components=4, random_state=42)
gmm.fit(scaled_data)

cluster_labels = gmm.predict(scaled_data)

In [ ]:
from collections import Counter

Counter(cluster_labels)

In [ ]:
data['cluster'] = cluster_labels

print(data.head())

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
from itertools import combinations

columns = ['#edges', 'weighted_degree_assortativity', 'avg_weighted_neighbor_degree', 'avg_neighbor_degree']

combinations_3d = list(combinations(columns, 3))

for combo in combinations_3d:
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    
    x, y, z = data[combo[0]], data[combo[1]], data[combo[2]]
    colors = data['cluster']
    
    scatter = ax.scatter(x, y, z, c=colors, cmap='viridis', s=50)
    
    ax.set_xlabel(combo[0])
    ax.set_ylabel(combo[1])
    ax.set_zlabel(combo[2])
    ax.set_title(f'3D Plot: {combo[0]} vs {combo[1]} vs {combo[2]}')
    
    legend1 = ax.legend(*scatter.legend_elements(), title="Cluster", loc="upper right")
    ax.add_artist(legend1)
    
    # Mostra il grafico
    plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
from itertools import combinations

columns = ['#edges', 'weighted_degree_assortativity', 'avg_weighted_neighbor_degree', 'avg_neighbor_degree']

combinations_2d = list(combinations(columns, 2))

for combo in combinations_2d:
    fig = plt.figure()
    ax = fig.add_subplot(111)
    
    x, y = data[combo[0]], data[combo[1]]
    colors = data['cluster']
    
    scatter = ax.scatter(x, y, c=colors, cmap='viridis', s=50)
    
    ax.set_xlabel(combo[0])
    ax.set_ylabel(combo[1])
    ax.set_title(f'2D Plot: {combo[0]} vs {combo[1]}')
    
    legend1 = ax.legend(*scatter.legend_elements(), title="Cluster", loc="upper right")
    ax.add_artist(legend1)
    
    # Mostra il grafico
    plt.show()

### Clustering Chris 2

In [ ]:
data = pd.read_csv('ColonieAdultaGephi/for_clustering.csv')
data2 = pd.read_csv('ColonieAdultaGephi/for_clustering_2.csv')
data_tot = pd.merge(data, data2, on='node')

In [ ]:
data_tot

In [ ]:
nodes = data_tot['node']
features = data_tot.drop(columns=['node'])
features = features.clip(lower=-1e10, upper=1e10)

from sklearn.preprocessing import StandardScaler

# Normalizzazione delle feature
scaler = StandardScaler()
normalized_features = scaler.fit_transform(features)

from sklearn.decomposition import PCA

pca = PCA(n_components=2)
reduced_features = pca.fit_transform(normalized_features)

In [ ]:
import seaborn as sns

# Visualizzazione dello spazio ridotto
plt.figure(figsize=(8, 6))
sns.scatterplot(x=reduced_features[:, 0], y=reduced_features[:, 1])
plt.title('Spazio ridotto con PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

In [ ]:
from sklearn.mixture import GaussianMixture

# Clustering con Gaussian Mixture Model
n_clusters = 4
gmm = GaussianMixture(n_components=n_clusters, random_state=42)
clusters = gmm.fit_predict(reduced_features)

In [ ]:
from sklearn.cluster import KMeans

# Clustering con KMeans
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(reduced_features)

In [ ]:
from sklearn.cluster import DBSCAN

# Clustering con DBSCAN
dbscan = DBSCAN(eps=0.7, min_samples=200)
clusters = dbscan.fit_predict(reduced_features)

In [ ]:
data_tot['cluster'] = clusters

In [ ]:
from sklearn.metrics import silhouette_score

# Valutazione del clustering
silhouette_avg = silhouette_score(normalized_features, clusters)
print(f'Silhouette Score: {silhouette_avg}')

In [ ]:
data_tot['cluster'].value_counts()

In [ ]:
# Visualizzazione dei cluster
plt.figure(figsize=(8, 6))
sns.scatterplot(x=reduced_features[:, 0], y=reduced_features[:, 1], hue=clusters, palette=['#fde725', '#440154', '#21918c'])
plt.title('Clustering con Gaussian Mixture Model')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(title='Cluster')
plt.show()

In [ ]:
# Analisi delle feature per cluster
cluster_analysis = data_tot.groupby('cluster').mean()
print("Caratteristiche medie per cluster:")
print(cluster_analysis)

### Clustering Chris 3

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations

In [ ]:
data = pd.read_csv('ColonieLarvaGephi/for_clustering.csv')

In [ ]:
data2 = pd.read_csv('ColonieLarvaGephi/for_clustering_2.csv')

In [ ]:
data_tot = pd.merge(data, data2, on='node')

In [ ]:
# Separazione di 'node' dalle feature
nodes = data_tot['node']
features = data_tot.drop(columns=['node'])

In [ ]:
# Normalizzazione delle feature
scaler = StandardScaler()
normalized_features = scaler.fit_transform(features)

In [ ]:
# Clustering preliminare per etichette fittizie (poiché ANOVA richiede un target, usiamo il clustering come proxy)
n_clusters = 4
gmm = GaussianMixture(n_components=n_clusters, random_state=42)
clusters = gmm.fit_predict(normalized_features)

In [ ]:
# Selezione delle feature con ANOVA (utilizziamo le etichette dei cluster come target temporaneo)
selector = SelectKBest(score_func=f_classif, k='all') # k='all' valuta tutte le feature
selector.fit(normalized_features, clusters)

In [ ]:
feature_scores = pd.DataFrame({
    'Feature': features.columns,
    'ANOVA_Score': selector.scores_
}).sort_values(by='ANOVA_Score', ascending=False)

print("Feature scores con ANOVA:")
print(feature_scores)

In [ ]:
selector.get_support(indices=True)[:N]

In [ ]:
# Selezioniamo le migliori N feature
N = 4
# best_features_idx = selector.get_support(indices=True)[:N]
best_features_idx = [0, 1, 3, 4, 5]
selected_features = features.iloc[:, best_features_idx]

In [ ]:
# Clustering con le feature selezionate
selected_normalized_features = normalized_features[:, best_features_idx]
gmm = GaussianMixture(n_components=n_clusters, random_state=42)
clusters = gmm.fit_predict(selected_normalized_features)

In [ ]:
data_tot['cluster'] = clusters

In [ ]:
# Visualizzazione di ogni coppia di feature
selected_feature_names = selected_features.columns
combs = list(combinations(selected_feature_names, 2))

In [ ]:
# Plot di ogni coppia di feature
for feature_pair in combs:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        x=data_tot[feature_pair[0]],
        y=data_tot[feature_pair[1]],
        hue=data_tot['cluster'],
        palette='viridis'
    )
    plt.title(f'Clustering su {feature_pair[0]} vs {feature_pair[1]}')
    plt.xlabel(feature_pair[0])
    plt.ylabel(feature_pair[1])
    plt.legend(title='Cluster')
    plt.show()

In [ ]:
# Valutazione del clustering
silhouette_avg = silhouette_score(selected_normalized_features, clusters)
print(f'Silhouette Score con feature selezionate: {silhouette_avg}')

### Clustering Chris 4

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.decomposition import PCA

In [ ]:
data = pd.read_csv('ColonieAdultaGephi/for_clustering.csv')

In [ ]:
data2 = pd.read_csv('ColonieAdultaGephi/for_clustering_2.csv')

In [ ]:
data_tot = pd.merge(data, data2, on='node')

In [ ]:
nodes = data_tot['node']
features = data_tot.drop(columns=['node'])

In [ ]:
features = features.clip(lower=-1e10, upper=1e10)

In [ ]:
# Normalizzazione
scaler = StandardScaler()
normalized_data = scaler.fit_transform(features)

In [ ]:
# PCA a 2 dimensioni
n_components = 2
pca = PCA(n_components=n_components)
pca_transformed = pca.fit_transform(normalized_data)

In [ ]:
# Trasformazione inversa (ricostruzione dei dati nel dominio fisico)
reconstructed_data = pca.inverse_transform(pca_transformed)
original_scale_data = scaler.inverse_transform(reconstructed_data)

reconstructed_df = pd.DataFrame(original_scale_data, columns=list(features.columns))

In [ ]:
from sklearn.cluster import KMeans

# Clustering con KMeans
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(original_scale_data)

reconstructed_df['cluster'] = clusters

In [ ]:
reconstructed_df['cluster'].value_counts()

In [ ]:
feature_list = list(features.columns)

# Numero di feature
num_features = len(feature_list)

# Creazione della figura e griglia di subplot
fig, axes = plt.subplots(num_features, num_features, figsize=(15, 15), constrained_layout=True)

# Loop su righe e colonne della griglia
for i, feature_x in enumerate(feature_list):
    for j, feature_y in enumerate(feature_list):
        ax = axes[i, j]

        if i == j:  # Se siamo sulla diagonale
            # Inserire un istogramma della feature
            sns.histplot(reconstructed_df[feature_x], bins=20, ax=ax, kde=True, color="gray")
            ax.set_title(f'Distribuzione di {feature_x}', fontsize=10)
        else:
            # Scatterplot per combinazioni diverse
            sns.scatterplot(
                x=reconstructed_df[feature_x],
                y=reconstructed_df[feature_y],
                hue=reconstructed_df['cluster'],
                palette='viridis',
                ax=ax,
                legend=False,
                s=10
            )

        # Etichettare solo il bordo sinistro e il fondo
        if i == num_features - 1:
            ax.set_xlabel(feature_x)
        else:
            ax.set_xlabel('')
            ax.tick_params(labelbottom=False)

        if j == 0:
            ax.set_ylabel(feature_y)
        else:
            ax.set_ylabel('')
            ax.tick_params(labelleft=False)

# Imposta una legenda complessiva fuori dalla griglia
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, title='Cluster', loc='upper right')

plt.suptitle('Scatterplot Pairwise delle Feature', fontsize=16)
plt.show()

In [ ]:
feature = 'clustering_coefficient'
min(reconstructed_df[feature])

In [ ]:
l = list(reconstructed_df[feature])
reconstructed_df[feature] += abs(min(l))
min(reconstructed_df[feature])

In [ ]:
# Visualizzazione per coppie di feature
feature_pairs = list(combinations(list(features.columns), 2))
for feature_pair in feature_pairs:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        x=reconstructed_df[feature_pair[0]],
        y=reconstructed_df[feature_pair[1]],
        hue=reconstructed_df['cluster'],
        palette='viridis'
    )
    plt.title(f'Clustering su {feature_pair[0]} vs {feature_pair[1]}')
    plt.xlabel(feature_pair[0])
    plt.ylabel(feature_pair[1])
    plt.legend(title='Cluster')
    plt.show()